# Associate the extracted places with Cliopatria polities

Builds the benchmark dataset: for each benchmark individual and life stage, one or more Cliopatria polities for the place extracted from the biography in `02_extract_places_from_biographies.ipynb`.

1. The Cliopatria polities whose territory holds each place during the stage years given by the biography are listed as candidates.
2. Claude Code agents choose among the candidates the polity or polities of each stage.

Agent outputs are cached in `benchmark300/cliopatria_association/`. The comparison with Cultura is in `04_benchmark_results.ipynb`.

In [ ]:
import json
import subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import polars as pl
from IPython.display import Markdown
from shapely import STRtree, make_valid, points
from shapely.geometry import shape
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent
FOLDER = ROOT / "scripts" / "5-annotations" / "benchmark" / "benchmark300"
PROMPTS = ROOT / "notebooks" / "notebooks_validation" / "prompts"
EXTRACTION = FOLDER / "extraction_life_stages"
ASSOCIATION = FOLDER / "cliopatria_association"
LOGS = FOLDER / "logs"
CLIOPATRIA = ROOT / "data" / "cliopatria_data" / "cliopatria_V2" / "cliopatria_polities_only_v3.geojson"
OUT = FOLDER / "benchmark_life_stage_polities.parquet"
MODEL = "claude-opus-5-5"
STAGES = ["pre_active", "active", "post_active"]
ASSOCIATION_GROUP = 40
WORKERS = 10
LIMIT = None
for folder in (ASSOCIATION, LOGS):
    folder.mkdir(exist_ok=True)

## Benchmark individuals

The individuals whose biography could be downloaded (`building_scripts/benchmark300_texts.py`), one text file each in `benchmark300/texts/`.

In [ ]:
people = pl.DataFrame([{key: person[key] for key in ("qid", "label", "dictionary_name", "biography_url")}
                       for person in json.loads((FOLDER / "benchmark300_texts.json").read_text(encoding="utf-8"))])
if LIMIT:
    people = people.head(LIMIT)
assert people["qid"].is_unique().all()
assert all((FOLDER / "texts" / f"{qid}.txt").exists() for qid in people["qid"])
print(f"{len(people)} individuals")
people.head()

### Claude Code runner

Each agent is a headless Claude Code session (`claude -p`) started from the repository root, with only the tools it needs; its full output is kept in `benchmark300/logs/`.

In [ ]:
def claude(prompt, tools, allowed, log):
    """Run one headless Claude Code session and return its final reply."""
    command = ["claude", "-p", "--model", MODEL, "--output-format", "json", *(["--allowedTools", *allowed] if allowed else []), "--tools", *tools]
    result = subprocess.run(command, input=prompt, cwd=ROOT, capture_output=True, text=True, timeout=3600)
    log.write_text(result.stdout + result.stderr, encoding="utf-8")
    return json.loads(result.stdout)["result"]


def groups(qids, size):
    return [qids[start:start + size] for start in range(0, len(qids), size)]


def done(path, qids):
    return path.exists() and set(qids) <= set(json.loads(path.read_text(encoding="utf-8")))


def run_all(task, jobs, label):
    """Run task(number, qids) for every job in parallel; a failed group is reported and left for the next run."""
    def safe(job):
        try:
            task(*job)
        except Exception as error:
            print(f"{label} group {job[0]:02d} failed: {error}")
    with ThreadPoolExecutor(WORKERS) as pool:
        list(tqdm(pool.map(safe, jobs), total=len(jobs), desc=label))

## Extracted places

The extraction of `02_extract_places_from_biographies.ipynb`.

In [ ]:
qids = set(people["qid"])
extraction = {qid: person for path in sorted(EXTRACTION.glob("agent_*.json"))
              for qid, person in json.loads(path.read_text(encoding="utf-8")).items() if qid in qids}
print(f"{len(extraction)} / {len(people)} individuals extracted")
filled = pl.DataFrame([{"stage": stage, "place": bool(person.get(stage)),
                        "years": bool(person.get(stage) and any(year is not None for year in person[stage].get("years") or []))}
                       for person in extraction.values() for stage in STAGES])
filled.group_by("stage").agg(individuals=pl.len(), with_place=pl.col("place").sum(), with_years=pl.col("years").sum()).sort(pl.col("stage").cast(pl.Enum(STAGES)))

## Step 1: Cliopatria candidates

For each stage, the Cliopatria polities whose territory contains a point of the extracted place and that existed during the stage years given by the biography. A missing bound is taken from the life years the biography gives for the other stages; a stage with no years at all keeps every polity that ever held the place. Each candidate comes with its years of existence and the polities it belongs to (`MemberOf` and `Components` of Cliopatria).

In [ ]:
features = json.loads(CLIOPATRIA.read_text())["features"]
polygons = STRtree(make_valid([shape(feature["geometry"]) for feature in features]))
properties = [feature["properties"] for feature in features]
parents = {}
for polity in properties:
    for parent in filter(None, polity["MemberOf"].split(";")):
        parents.setdefault(polity["Name"], set()).add(parent)
    for child in filter(None, polity["Components"].split(";")):
        parents.setdefault(child, set()).add(polity["Name"])
print(f"{len(features):,} Cliopatria polygons, {len({polity['Name'] for polity in properties}):,} polities")

In [ ]:
def stage_years(person, stage):
    known = [year for name in STAGES if person.get(name) for year in person[name].get("years") or [] if year is not None]
    start, end = (list(person[stage].get("years") or []) + [None, None])[:2]
    if known:
        start = min(known) if start is None else start
        end = max(known) if end is None else end
    return start, end


def candidates(person, stage):
    """Cliopatria polities holding a point of the stage place during the stage years."""
    spots = person[stage].get("points") or []
    if not spots:
        return []
    start, end = stage_years(person, stage)
    found = {}
    for spot, index in zip(*polygons.query(points([(longitude, latitude) for _, longitude, latitude in spots]), predicate="within")):
        polity = properties[index]
        if start is not None and (polity["ToYear"] < start or polity["FromYear"] > end):
            continue
        entry = found.setdefault(polity["Name"], {"name": polity["Name"], "from": polity["FromYear"], "to": polity["ToYear"],
                                                  "member_of": sorted(parents.get(polity["Name"], ())), "points": []})
        entry["from"], entry["to"] = min(entry["from"], polity["FromYear"]), max(entry["to"], polity["ToYear"])
        if spots[spot][0] not in entry["points"]:
            entry["points"].append(spots[spot][0])
    return sorted(found.values(), key=lambda entry: entry["from"])


association_input = {}
for qid, person in extraction.items():
    for stage in STAGES:
        if person.get(stage):
            found = candidates(person, stage)
            if found:
                association_input.setdefault(qid, {})[stage] = {
                    "place": person[stage]["place"], "written": person[stage]["written"], "written_en": person[stage]["written_en"],
                    "years": person[stage].get("years"), "extracts": person[stage]["verbatim_en"], "candidates": found}
counts = pl.DataFrame([{"stage": stage, "candidates": len(entry["candidates"])} for stages in association_input.values() for stage, entry in stages.items()])
print(f"{len(association_input)} individuals with at least one stage to associate")
counts.group_by("stage").agg(stages=pl.len(), median_candidates=pl.col("candidates").median(), max_candidates=pl.col("candidates").max()).sort(pl.col("stage").cast(pl.Enum(STAGES)))

## Step 2: polity association

### Association prompt

In [ ]:
ASSOCIATION_PROMPT = (PROMPTS / "biography_cliopatria_association.md").read_text(encoding="utf-8").split("---", 1)[1]
Markdown(ASSOCIATION_PROMPT)

### Run the association agents

Groups of `ASSOCIATION_GROUP` individuals; the candidates are written in the prompt, so the agents have no tools. A polity name that is not one of the stage candidates is dropped.

In [ ]:
def associate(number, qids):
    path = ASSOCIATION / f"agent_{number:02d}.json"
    if not done(path, qids):
        prompt = ASSOCIATION_PROMPT.replace("{CANDIDATES}", json.dumps({qid: association_input[qid] for qid in qids}, ensure_ascii=False, indent=1))
        reply = claude(prompt, [""], [], LOGS / f"association_{number:02d}.log")
        answer = json.loads(reply[reply.index("{"):reply.rindex("}") + 1])
        path.write_text(json.dumps(answer, ensure_ascii=False, indent=2), encoding="utf-8")


association_jobs = list(enumerate(groups(sorted(association_input), ASSOCIATION_GROUP)))
run_all(associate, association_jobs, "association")
missing = [number for number, qids in association_jobs if not done(ASSOCIATION / f"agent_{number:02d}.json", qids)]
print(f"{len(association_jobs) - len(missing)} / {len(association_jobs)} groups associated; missing: {missing}")

In [ ]:
association = {qid: stages for path in sorted(ASSOCIATION.glob("agent_*.json"))
               for qid, stages in json.loads(path.read_text(encoding="utf-8")).items() if qid in association_input}
dropped = []
for qid, stages in association.items():
    for stage, answer in stages.items():
        allowed = {entry["name"] for entry in association_input[qid].get(stage, {}).get("candidates", [])}
        dropped += [(qid, stage, name) for name in answer.get("polities", []) if name not in allowed]
        answer["polities"] = [name for name in answer.get("polities", []) if name in allowed]
print(f"{len(association)} / {len(association_input)} individuals associated; {len(dropped)} polity names not among the candidates dropped")
dropped[:10]

## Final benchmark dataset

One row per individual and stage, with the place and years from the biography and one or more Cliopatria polities (brackets of Cliopatria names removed, as in Cultura). Saved to `benchmark300/benchmark_life_stage_polities.parquet`.

In [ ]:
rows = []
for qid, person in extraction.items():
    for stage in STAGES:
        extracted = person.get(stage) or {}
        answer = association.get(qid, {}).get(stage) or {}
        start, end = (extracted.get("years") or [None, None]) if extracted else (None, None)
        rows.append({
            "qid": qid, "stage": stage, "biography_start_year": start, "biography_end_year": end,
            "place": extracted.get("place"), "place_as_written": extracted.get("written"), "place_as_written_english": extracted.get("written_en"),
            "verbatim": extracted.get("verbatim", []), "verbatim_english": extracted.get("verbatim_en", []),
            "coordinates": [f"{name} ({latitude}, {longitude})" for name, longitude, latitude in extracted.get("points") or []],
            "extraction_confidence": extracted.get("confidence"), "extraction_reasoning": extracted.get("reasoning"),
            "candidate_polities": [entry["name"].strip("()") for entry in association_input.get(qid, {}).get(stage, {}).get("candidates", [])],
            "polities": [name.strip("()") for name in answer.get("polities", [])],
            "association_confidence": answer.get("confidence"), "association_reasoning": answer.get("reasoning"), "model_name": MODEL,
        })
benchmark = (pl.DataFrame(rows, schema_overrides={"biography_start_year": pl.Int64, "biography_end_year": pl.Int64})
             .join(people, on="qid", how="left").sort("qid", pl.col("stage").cast(pl.Enum(STAGES))))
assert benchmark.select("qid", "stage").is_unique().all()
assert len(benchmark) == 3 * len(extraction)
benchmark.write_parquet(OUT)
print(f"{len(benchmark)} rows ({benchmark['qid'].n_unique()} individuals) saved to {OUT.relative_to(ROOT)}")
benchmark.filter(pl.col("polities").list.len() > 0).sample(5, seed=42).select("label", "stage", "biography_start_year", "biography_end_year", "place", "polities", "association_confidence")

### Polities per stage

Stages with no polity (no place in the biography, or no Cliopatria polity for it), one polity, or several.

In [ ]:
(benchmark.with_columns(n=pl.col("polities").list.len())
 .group_by("stage").agg(no_polity=(pl.col("n") == 0).sum(), one_polity=(pl.col("n") == 1).sum(), several_polities=(pl.col("n") > 1).sum())
 .sort(pl.col("stage").cast(pl.Enum(STAGES))))